In [333]:
import pandas as pd
import numpy as np
import re
import difflib

In [334]:
cps = pd.read_csv(r'cellphones_full.csv')
cps.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Tên                    966 non-null    str  
 1   Giá                    966 non-null    str  
 2   Link                   966 non-null    str  
 3   Kích thước màn hình    865 non-null    str  
 4   Công nghệ màn hình     806 non-null    str  
 5   Camera sau             847 non-null    str  
 6   Camera trước           817 non-null    str  
 7   Chipset                850 non-null    str  
 8   Công nghệ NFC          763 non-null    str  
 9   Bộ nhớ trong           913 non-null    str  
 10  Thẻ SIM                695 non-null    str  
 11  Hệ điều hành           756 non-null    str  
 12  Độ phân giải màn hình  657 non-null    str  
 13  Tính năng màn hình     725 non-null    str  
 14  Loại CPU               586 non-null    str  
 15  Dung lượng RAM         870 non-null    str  
 16  P

In [335]:
feature_mapping = {
    "Tên": "Name",
    "Giá": "Price",
    "Link": "Link",
    "Kích thước màn hình": "Screen Size",
    "Công nghệ màn hình": "Display",
    "Camera sau": "Rear Camera",
    "Camera trước": "Front Camera",
    "Chipset": "Chipset",
    "Công nghệ NFC": "NFC",
    "Bộ nhớ trong": "ROM",
    "Thẻ SIM": "SIM Card",
    "Hệ điều hành": "Operating System",
    "Độ phân giải màn hình": "Screen Resolution",
    "Tính năng màn hình": "Display Features",
    "Loại CPU": "CPU",
    "Dung lượng RAM": "RAM",
    "Pin": "Battery",
    "Tương thích": "Compatibility",
    "Cảm biến": "Sensors",
}

cps = cps.rename(columns=feature_mapping)


In [336]:
df = cps.copy()

In [337]:
antutu = pd.read_csv(r'antutu_score_socket.csv')
att = antutu.copy()

In [338]:
def extract_refresh_rate(raw_value):
    if pd.isna(raw_value):
        return 0

    s = str(raw_value).lower().strip()
    
    match = re.search(r'([\d.]+)\s*hz', s)
    return float(match.group(1)) if match else 0


In [339]:
df["Refresh Rate"] = df["Display Features"].apply(extract_refresh_rate)

CLEAN NAME

In [340]:
def clean_phone_name(raw_name):
    if pd.isna(raw_name):
        return ""

    name = str(raw_name).lower().strip()
    if not name:
        return ""

    # Chuẩn hóa dấu nối và loại bỏ phân đoạn không cần thiết
    name = re.sub(r"[\u2010\u2013\u2014\u2212]", "-", name)
    name = re.sub(r"\s*\|\s*.*$", "", name)
    name = re.sub(r"\bđiện thoại\b", "", name, flags=re.I)
    name = re.sub(r"\b(?:ram|rom)\b", "", name, flags=re.I)

    # Xóa các cụm từ quảng cáo / danh mục không phải model
    patterns_to_delete = [
        r"chính hãng",
        r"vn/?a",
        r"bản quốc tế",
        r"bản chính hãng",
        r"xách tay",
        r"nhập khẩu",
        r"full ?box",
        r"open ?box",
        r"like ?new",
        r"second ?hand",
        r"trả góp",
        r"giá tốt",
        r"giá rẻ",
        r"hàng chính hãng",
        r"hàng.*",
        r"Exynos",
        r"Snapdragon",
        r"special edition",
        r"edition",
        r'china',
        '2021',
        '2022',
        '2023',
        '2024',
        '2025',
        
    ]
    name = re.sub(r"\b(?:" + "|".join(patterns_to_delete) + r")\b", "", name, flags=re.I)

    # Xóa các thông tin mạng và kết nối không phải tên model
    name = re.sub(r"\b(?:4g|5g|nfc|lte|wifi|bluetooth)\b", "", name, flags=re.I)

    # Xóa dung lượng RAM/ROM/ổ cứng
    name = re.sub(r"\b\d+(?:[\.,]\d+)?\s*(?:gb|tb|mb)\b", "", name, flags=re.I)
    name = re.sub(r"\b\d+\s*[x×]\s*\d+\s*(?:gb|tb|mb)\b", "", name, flags=re.I)
    name = re.sub(r"\b\d+\s*[+\/]\s*\d+\s*(?:gb|tb|mb)\b", "", name, flags=re.I)

    # Loại bỏ ký tự không cần và chuẩn hóa khoảng trắng
    name = re.sub(r"[\[\]\(\)\{\}]", " ", name)
    name = re.sub(r"[^\w\s\-]+", " ", name)
    name = re.sub(r"\s{2,}", " ", name)
    name = re.sub(r"\b-\b", " ", name)
    name = name.strip()

    return name

In [341]:
df["Name"] = df["Name"].apply(clean_phone_name)

In [342]:
def count_matching_names(df1, col1, df2, col2):
    set1 = set(df1[col1].unique())
    set2 = set(df2[col2].unique())
    
    matching = set1 & set2
    
    return {
        'matches': len(matching),
        'df1_unique': len(set1),
        'df2_unique': len(set2),
        'match_rate_df1': f"{(len(matching) / len(set1) * 100):.1f}%",
        'match_rate_df2': f"{(len(matching) / len(set2) * 100):.1f}%",
        'matching_names': sorted(list(matching))
    }

CLEAN PRICE

In [343]:
def clean_price(raw_price):
    if pd.isna(raw_price):
        return 0
    s = str(raw_price).lower().strip()
    if re.search(r'liên hệ', s):
        return 0
    
    s = re.sub(r'[đ]', '', s)
    s = s.replace('.', '')

    match = re.search(r'(\d+)', s)
    return int(match.group(1)) if match else 0


In [344]:
df["Price"] = df["Price"].apply(clean_price)

CLEAN STORAGE

In [345]:
def clean_storage(raw_value):
    if pd.isna(raw_value):
        return 0

    s = str(raw_value).lower().strip()
    
    match = re.search(r'([\d.]+)', s)
    if not match:
        return 0
    
    value = float(match.group(1))
    
    if 'tb' in s:
        value = value * 1024
    if 'mb' in s:
        value = value / 1024
    
    return value


In [346]:
df["RAM"] = df["RAM"].apply(clean_storage)
df["ROM"] = df["ROM"].apply(clean_storage)

CLEAN SIZE, BATTERY

In [348]:
def clean_metrics(raw_value):
    if pd.isna(raw_value):
        return 0.0

    s = str(raw_value).lower().strip()
    
    match = re.search(r'([\d.]+)', s)
    return float(match.group(1)) if match else 0.0

In [349]:
cols_to_clean = ["Screen Size","Battery"]
for col in cols_to_clean:
    df[col] = df[col].apply(clean_metrics)

CLEAN CPU


In [350]:
WORD_CORE_MAP = {
    'dual': 2, 'quad': 4, 'hexa': 6, 'octa': 8,
    'deca': 10, 'nona': 9,
    'tám nhân': 8, 'lõi tám': 8, 'tám lõi': 8,
    'lõi tứ': 4, 'lõi đơn': 1,
}

def normalize(t):
    t = str(t).lower()
    t = re.sub(r'(\d)[,،，](\d)', r'\1.\2', t)
    t = t.replace('，', ',')
    return t

def extract_groups(t):
    patterns = [
        r'(\d+)\s*[x×*]\s*(\d+\.?\d*)\s*ghz',
        r'(\d+)\s*[x×*]\s*[\w\s.\-]+?@\s*(\d+\.?\d*)\s*ghz',
        r'(\d+)\s*[x×*]\s*[\w\s.\-]+?(?:up to|tối đa|lên đến|đến)\s*(\d+\.?\d*)\s*ghz',
        r'(\d+)\s*[x×]\s*[\w\s.\-]+?\((?:tối đa\s*)?(\d+\.?\d*)\s*ghz\)',
        r'(\d+)\s*(?:nhân|lõi)\s+(\d+\.?\d*)\s*ghz',
        r'(\d+)\s*[x×]\s*[a-z]\w+\s+(\d+\.?\d*)\s*ghz',
    ]
    for pat in patterns:
        found = re.findall(pat, t)
        if len(found) >= 2:
            valid = [(int(m[0]), float(m[-1])) for m in found if float(m[-1]) > 0.5]
            if len(valid) >= 2:
                return valid
    return []

def extract_single_freq(t):
    freqs = re.findall(r'(\d+\.?\d*)\s*ghz', t)
    freqs = [float(f) for f in freqs if float(f) > 0.5]
    return max(freqs) if freqs else np.nan

def extract_cpu_features(raw):
    if pd.isna(raw) or str(raw).strip() == '':
        return {}
    t = normalize(raw)
    result = {}

    for word, num in WORD_CORE_MAP.items():
        if word in t:
            result['num_cores'] = num
            break
    if 'num_cores' not in result:
        m = re.search(r'(\d+)\s*(nhân|cores?|lõi)', t)
        result['num_cores'] = int(m.group(1)) if m else np.nan

    groups = extract_groups(t)
    if groups:
        groups_sorted = sorted(groups, key=lambda x: x[1], reverse=True)
        result['perf_cores']    = groups_sorted[0][0]
        result['perf_freq_ghz'] = groups_sorted[0][1]
        result['eff_cores']     = groups_sorted[-1][0]
        result['eff_freq_ghz']  = groups_sorted[-1][1]
        # Cộng tổng từ groups nếu num_cores chưa có
        if pd.isna(result.get('num_cores')):
            result['num_cores'] = sum(g[0] for g in groups)
    else:
        single = extract_single_freq(t)
        if not np.isnan(single):
            result['max_freq_ghz'] = single

    if 'perf_cores' not in result:
        m = re.search(r'(\d+)\s*lõi\s*(?:hiệu năng|hiệu suất)', t)
        if m: result['perf_cores'] = int(m.group(1))
    if 'eff_cores' not in result:
        m = re.search(r'(\d+)\s*lõi\s*(?:tiết kiệm|nhỏ)', t)
        if m: result['eff_cores'] = int(m.group(1))

    return result


In [351]:
features = df['CPU'].apply(extract_cpu_features)
cpu_df   = pd.json_normalize(features)
df       = pd.concat([df.reset_index(drop=True), cpu_df], axis=1)

CLEAN OPERATING SYSTEM

In [352]:
def advanced_clean_os(text):
    if (
        pd.isna(text)
        or not isinstance(text, str)
        or "cập nhật" in text.lower()
    ):
        return 1, "Android", None

    text = text.strip()
    
    is_android = 1
    os_name = "Android"
    if "ios" in text.lower():
        is_android = 0
        os_name = "iOS"

    os_version = None

    if text.isdigit():
        return is_android, os_name, float(text)

    if "nâng cấp" in text.lower():
        upgraded_version = re.findall(r"Android\s*(\d+(?:\.\d+)?)", text)
        if upgraded_version:
            return is_android, os_name, float(upgraded_version[-1])

    version_match = re.search(r"(?:Android|iOS)\s*(\d+(?:\.\d+)?)", text, re.I)
    if version_match:
        os_version = float(version_match.group(1))
    else:
        os_version = None

    return is_android, os_name, os_version

In [353]:
df["OS_Is_Android"], df["OS_Name"], \
    df["OS_Version"] = zip(*df["Operating System"].apply(advanced_clean_os))

CLEAN RESOLUTION

In [354]:
def extract_res_row(text):
    # Nếu dòng bị trống (NaN) hoặc không phải chữ
    if pd.isna(text) or not isinstance(text, str):
        return None, None

    # Tìm cấu trúc a x b ở đầu dòng
    match = re.search(r"^(\d+)\s*[xX×]\s*(\d+)", text.strip())

    if match:
        # Trả về một Tuple gồm (Width, Height) kiểu số nguyên
        return int(match.group(1)), int(match.group(2))

    return None, None

In [355]:
df["Reso_Width"], df["Reso_Height"] = zip(
    *df["Screen Resolution"].apply(extract_res_row)
)

In [356]:
def clean_sim_options(text):
    max_nano = 0
    max_esim = 0
    max_micro = 0
    max_mini = 0

    if pd.isna(text) or not isinstance(text, str):
        return max_nano, max_esim, max_micro, max_mini

    text_lower = text.lower()

    options = re.split(r"hoặc|/|;", text_lower)

    for option in options:
        option = option.strip()

        nano_in_opt = 0
        esim_in_opt = 0

        # Xử lý eSIM 
        if "esim" in option:
            match_esim = re.search(r"(\d+)\s*esim", option)
            if match_esim:
                esim_in_opt = int(match_esim.group(1))
            elif "dual" in option or "kép" in option:
                esim_in_opt = 2
            else:
                esim_in_opt = 1

        # Xử lý Nano SIM 
        if "nano" in option or "sim 1 + sim 2" in option:
            match_nano = re.search(r"(\d+)\s*nano", option)
            if match_nano:
                nano_in_opt = int(match_nano.group(1))
            elif (
                "dual" in option
                or "kép" in option
                or "sim 1 + sim 2" in option
            ):
                nano_in_opt = 2
            else:
                nano_in_opt = 1
        elif "2 sim" in option and "nano" in text_lower:
            nano_in_opt = 2
        # Trường hợp ghi mỗi chữ "Nano-SIM" thuần túy
        elif "nano" in option:
            nano_in_opt = 1

        # Cập nhật giá trị max
        max_nano = max(max_nano, nano_in_opt)
        max_esim = max(max_esim, esim_in_opt)

        # Xử lý các loại SIM cổ (Mini, Micro)
        if "micro" in option:
            max_micro = 1
        if "mini" in option:
            max_mini = 1

    return max_nano, max_esim, max_micro, max_mini

In [357]:
(
    df["Nano_SIM_Count"],
    df["eSIM_Count"],
    df["Micro_SIM_Count"],
    df["Mini_SIM_Count"],
) = zip(*df["SIM Card"].apply(clean_sim_options))

In [358]:
df[['CPU', 'num_cores', 'perf_cores', 'eff_cores', 'perf_freq_ghz', 'eff_freq_ghz', 'max_freq_ghz']]

,CPU,num_cores,perf_cores,eff_cores,perf_freq_ghz,eff_freq_ghz,max_freq_ghz
0,CPU 6 lõi với 2 lõi hiệu năng và 4 lõi tiết ki...,6.0,2.0,4.0,NaN,NaN,NaN
1,8 nhân,8.0,NaN,NaN,NaN,NaN,NaN
2,CPU 6 lõi với 2 lõi hiệu năng và 4 lõi tiết ki...,6.0,2.0,4.0,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
961,2 nhân 2.5 GHz & 6 nhân 2.0 GHz,2.0,2.0,6.0,2.5,2.0,NaN
962,1×Cortex-A710 2.5GHz + 3×Cortex-A710 2.36GHz +...,NaN,NaN,NaN,NaN,NaN,2.5
963,NaN,NaN,NaN,NaN,NaN,NaN,NaN
964,NaN,NaN,NaN,NaN,NaN,NaN,NaN


CLEAN CHIPSET

In [359]:
_TRASH_VALUES = {
    "",
    "đang cập nhật",
    "mediatek",
    "exynos",
    "snapdragon",
    "bộ xử lý octa-core",
    "asr",
    "asr platform",
    "sc6531e",
    "ums9117",
}
_BRAND_PREFIXES = [
    r"qualcomm\s+(sm|sdm|msm|qm)\w+\s+",   # "Qualcomm SM8350 Snapdragon..." -> "Snapdragon..."
    r"qualcomm\s+",
    r"mediatek\s+",
    r"hisilicon\s+",
    r"samsung\s+",
    r"google\s+",
    r"huawei\s+",
    r"spreadtrum\s+",
    r"unisoc\s+",
    r"apple\s+",
    r"chip\s+",                      
]

def normalize_chipset(text):
    if not isinstance(text, str):
        return ""

    s = text.strip().lower()

    s = re.sub(r"\bthế\s*hệ\b", "gen", s) #"thế hệ" -> "gen"
    s = re.sub(r"\(.*?\)", "", s) #nội dung trong ()
    s = re.sub(r"(\w)\+", r"\1 plus", s) #từ + -> plus
    s = re.sub(r"[®™°•·]", " ", s) #Ký hiệu đặc biệt -> dấu cách
    s = re.sub(r"\b(sm|sdm|msm|apl)\w+\b", "", s) #(sm8350, sdm845, msm8998, apl0698...)

    for pat in _BRAND_PREFIXES:
        s = re.sub(rf"^{pat}", "", s)

    s = re.sub(r"\b(dành cho|cho|danh cho)\s+galaxy\b.*$", "", s)   # "dành cho Galaxy ..."
    s = re.sub(r"\bfor\s+galaxy\b.*$", "", s)                        # "for Galaxy ..."
    s = re.sub(r"\b\d+\s*nhân\b", "", s)                             # "8 nhân", "6 nhân"
    s = re.sub(r"\bocta[\s-]?core\b", "", s)                         # "octa core", "octa-core"
    s = re.sub(r"\b(mobile\s+)?platform\b", "", s)                   # "Mobile Platform"
    s = re.sub(r"\baccelerated\s+edition\b", "", s)                  # "Accelerated Edition"
    s = re.sub(r"\bflagship\b", "", s)                               # "Flagship"
    s = re.sub(r"\btối\s+đa\s+[\d.,]+\s*ghz\b", "", s)              # "tối đa 2.2GHz"
    s = re.sub(r"\btiến\s*trình\b.*$", "", s)                       # "tiến trình 4nm ..."
    s = re.sub(r"\btăng\s+lên\b.*$", "", s)                         # "tăng lên 42% AI ..."
    s = re.sub(r"\b5g\b", "", s)                                     # "5G"
    s = re.sub(r"\b4g\b", "", s)                                     # "4G"

    s = re.sub(r"\b\d+\s*nm\+?\b", "", s) #"6 nm"
    s = s.replace("-", " ")
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    if s in _TRASH_VALUES or len(s) < 3:
        return ""

    return s

In [360]:
def map_chipset_info(df_a, df_c) -> pd.DataFrame:

    map = {}
    for _, row in df_a.iterrows():
        norm = normalize_chipset(row["Chipset"])
        map[norm] = {"antutu_11": row["Antutu_11"], "clock": row["Clock"], "gpu": row["GPU"]}
 
    # Map vào từng dòng
    mapped = df_c["Chipset"].apply(lambda x : map.get(normalize_chipset(x)))

    df_out = df_c.copy()
    df_out["antutu_11"] = mapped.apply(lambda x: x["antutu_11"] if x else None)
    df_out["clock"] = mapped.apply(lambda x: x["clock"] if x else None)
    df_out["gpu"] = mapped.apply(lambda x: x["gpu"] if x else None)
 
    return df_out

In [361]:
df = map_chipset_info(att, df)

CLEAN NFC

In [362]:
df['NFC'] = df['NFC'].map(lambda x : 1 if x == "Có" else 0)

CLEAN CAMERA

In [363]:
def clean(text):
    # "hỗ trợ chụp 24MP hoặc 48MP"
    text = re.sub(r'hỗ trợ chụp.*?(?=\D{3}|$)', '', text, flags=re.IGNORECASE)
    # "(24MP và 48MP)", "(24MP hoặc 48MP)"
    text = re.sub(r'\([\d.]+\s*MP\s*(?:và|hoặc|or)\s*[\d.]+\s*MP\)', '', text, flags=re.IGNORECASE)
    # "hoặc 48MP" còn sót
    text = re.sub(r'(?:hoặc|hoac|hay|or)\s+[\d.]+\s*MP', '', text, flags=re.IGNORECASE)
    return text

In [364]:
def extract_mp_values(text):
    text = clean(text)
    vals  = re.findall(r'([\d.]+)\s*(?:MP|megapixel)', text, re.IGNORECASE)
    vals += re.findall(r'([\d.]+)M(?=[^a-zA-Z]|$)', text)
    return [float(v) for v in vals if v.count('.') <= 1 and float(v) >= 0.3]

In [365]:
def extract_aperture(text):
    vals   = re.findall(r'[fƒ]\s*/?\s*([\d.]+)', text, re.IGNORECASE)
    floats = [float(v) for v in vals if v.count('.') <= 1 and 0.5 <= float(v) <= 6.0]
    return min(floats) if floats else 0

In [366]:
def count_cameras(text: str, mps: list):
    if len(mps) >= 2:
        return len(mps)
    m = re.search(r'(\d)\s*camera', text, re.IGNORECASE)
    if m:
        return int(m.group(1))
    return 1 if mps else 0

In [367]:
def parse_rear(text):
    if not isinstance(text, str) or not text.strip():
        return {"rear_count": 0, "rear_mp_max": 0, "rear_f/": 0, "rear_ois": 0, "rear_telephoto": 0, "rear_wide": 0}
    mps      = extract_mp_values(text)
    aperture = extract_aperture(text)
    return {
        "rear_count": count_cameras(text, mps),
        "rear_mp_max": max(mps) if mps else 0,
        "rear_f/": aperture if aperture else 0,
        "rear_ois": int(bool(re.search(r'\bOIS\b', text, re.IGNORECASE))),
        "rear_telephoto": int(bool(re.search(r'tele(?:photo)?|zoom quang|kính tiềm vọng|periscope', text, re.IGNORECASE))),
        "rear_wide": int(bool(re.search(r'siêu rộng|ultra.?wide|góc rộng|wide|superwide', text, re.IGNORECASE))),
    }

In [368]:
def parse_front(text):
    if not isinstance(text, str) or not text.strip():
        return {"front_mp": 0, "front_f/": 0}
    mps = extract_mp_values(text)
    aperture = extract_aperture(text)
    return {
        "front_mp": max(mps) if mps else 0,
        "front_f/": aperture if aperture else 0,
    }

In [369]:
def parse_camera(df):
    rear  = df["Rear Camera"].apply(parse_rear).apply(pd.Series)
    front = df["Front Camera"].apply(parse_front).apply(pd.Series)
    df_out = pd.concat([df, rear, front], axis=1)

    return df_out

In [370]:
df = parse_camera(df)

In [371]:
# # Bước 1: Tách cột đó ra khỏi DataFrame trước (để tránh bị trùng lặp)
# score_col = df.pop('Camera_score')

# # Bước 2: Chèn lại vào vị trí mong muốn (Ví dụ: loc=2 là cột thứ 3 trong bảng)
# df.insert(loc=46, column='Camera_score', value=score_col)

Encode OS name. 1: Android and 0: iOS

In [374]:
# Nếu là Android thì thành 1, ngược lại (iOS) thì thành 0
df['OS_Name'] = (df['OS_Name'] == 'Android').astype(int)

Encode Display. 1: OLED/AMOLED, 2: IPS LCD/IPS/LCD, 0: other

In [375]:
display_lower = df['Display'].astype(str).str.lower()

# 2. Định nghĩa các điều kiện (Conditions)
conditions = [
    display_lower.str.contains('oled|amoled', regex=True),  # Nhóm 2
    display_lower.str.contains('ips lcd|ips|lcd', regex=True)      # Nhóm 1
]

# 3. Định nghĩa giá trị tương ứng cho từng điều kiện
choices = [2, 1]

# 4. Sử dụng np.select, mặc định (default) không khớp cái nào sẽ là 0
df['Display'] = np.select(conditions, choices, default=0)

Encode Chipset. Apple: 0, Snapdragon: 1, MediaTek: 2, Exynos: 3, Kirin: 4, Unisoc: 5, Other: 6

In [376]:
def encode_chipset(val):
    val = str(val).lower()
    if 'apple' in val or 'chip a' in val or 'bionic' in val:
        return 'Apple'
    elif 'snapdragon' in val or 'qualcomm' in val:
        return 'Snapdragon'
    elif 'dimensity' in val or 'helio' in val or 'mediatek' in val:
        return 'MediaTek'
    elif 'exynos' in val:
        return 'Exynos'
    elif 'kirin' in val:
        return 'Kirin'
    elif 'unisoc' in val:
        return 'Unisoc'
    else:
        return 'Other'
 
brand_map = {'Apple': 0, 'Snapdragon': 1, 'MediaTek': 2,
             'Exynos': 3, 'Kirin': 4, 'Unisoc': 5, 'Other': 6}
df['Chipset'] = df['Chipset'].apply(encode_chipset).map(brand_map)

Encode GPU

In [377]:
def encode_gpu(val):
    val = str(val).lower()
    if 'adreno'     in val: return 'Adreno'
    elif 'mali'     in val: return 'Mali'
    elif 'apple'    in val: return 'Apple GPU'
    elif 'xclipse'  in val: return 'Xclipse'
    elif 'powervr'  in val or 'img' in val: return 'PowerVR'
    elif 'immortalis' in val: return 'Immortalis'
    elif 'maleoon'  in val: return 'Maleoon'
    else: return 'Other'
 
gpu_map = {'Adreno': 0, 'Mali': 1, 'Apple GPU': 2, 'Xclipse': 3,
           'PowerVR': 4, 'Immortalis': 5, 'Maleoon': 6, 'Other': 7}
df['gpu_family_enc'] = df['gpu'].apply(encode_gpu).map(gpu_map)

Encode Reso

In [378]:
median_screen = df.loc[df['Screen Size'] > 0, 'Screen Size'].median()
df['Screen Size'] = df['Screen Size'].replace(0, median_screen)
 
df['PPI'] = (
    np.sqrt(df['Reso_Width']**2 + df['Reso_Height']**2) / df['Screen Size']
).round(1)

Derive PPI

In [379]:
median_screen = df.loc[df['Screen Size'] > 0, 'Screen Size'].median()
df['Screen Size'] = df['Screen Size'].replace(0, median_screen)

df['PPI'] = (
    np.sqrt(df['Reso_Width']**2 + df['Reso_Height']**2) / df['Screen Size']
).round(1)

df.drop(columns=['Reso_Width', 'Reso_Height'], inplace=True)


Derive SIM_total

In [380]:
df['SIM_total'] = (
    df['Nano_SIM_Count'] +
    df['eSIM_Count'] +
    df['Micro_SIM_Count'] +
    df['Mini_SIM_Count']
)

df.drop(columns=['Nano_SIM_Count', 'Micro_SIM_Count', 'Mini_SIM_Count'], inplace=True)



Derive eSIM, dien thoai nao co eSIM: 1, khong co: 0

In [381]:
df['has_eSIM'] = (df['eSIM_Count'] > 0).astype(int)

df.drop(columns=['eSIM_Count'], inplace=True)


In [382]:
camera = pd.read_csv('camera_score.csv')
camera = camera.rename(columns={'camera_score': 'Camera_score'})

# Fix 1: Drop dòng NaN name trước khi xử lý
camera = camera.dropna(subset=['name'])

# Fix 2: Thêm .strip() để tránh khoảng trắng thừa
camera['name'] = camera['name'].apply(clean_phone_name)
camera['chip'] = camera['chip'].apply(encode_chipset).map(brand_map)

# Encode chipset bên df để so sánh được với camera
df['Chipset_encoded'] = df['Chipset']

def get_camera_score(row):
    # Fix 2: Thêm .strip() ở đây nữa
    name = str(row['Name']).strip().lower()
    matches = camera[camera['name'] == name]

    if matches.empty:
        return None

    if len(matches) == 1:
        return matches.iloc[0]['Camera_score']

    # Nhiều chip: thử match theo chip
    chipset = row['Chipset_encoded']
    if pd.isna(chipset):
        return matches.iloc[0]['Camera_score']

    chip_match = matches[matches['chip'] == chipset]
    if not chip_match.empty:
        return chip_match.iloc[0]['Camera_score']
    else:
        return matches.iloc[0]['Camera_score']

df['Camera_score'] = df.apply(get_camera_score, axis=1)
df = df.drop(columns=['Chipset_encoded'])

In [384]:
# Helper function
def fill_median_group(df, col, group_col):
    df[col] = df.groupby(group_col)[col].transform(
        lambda x: x.fillna(x.median())
    )
    df[col] = df[col].fillna(df[col].median())  # fallback
    return df

# ── Nhóm CPU: groupby theo chipset_brand_enc ──────────────────
for col in ['perf_cores', 'eff_cores', 'perf_freq_ghz', 'eff_freq_ghz']:
    df = fill_median_group(df, col, 'Chipset')

# ── OS_Version: groupby theo os_is_ios ───────────────────────
df = fill_median_group(df, 'OS_Version', 'OS_Name')

# ── PPI: groupby theo display_tier ───────────────────────────
df = fill_median_group(df, 'PPI', 'Display')

# Kiểm tra
print(df[['perf_cores','eff_cores','perf_freq_ghz','eff_freq_ghz'
          ,'OS_Version','Camera_score','PPI']].isnull().sum())

perf_cores         0
eff_cores          0
perf_freq_ghz      0
eff_freq_ghz       0
OS_Version         0
Camera_score     790
PPI                0
dtype: int64


In [386]:
drop = [
    'Link', 'Rear Camera', 'Front Camera', 'CPU', 'Display Features', 'Screen Resolution',
    'Operating System', 'SIM Card', 'clock', 'OS_Is_Android', 'num_cores', 'Compatibility', 'Sensors', 'max_freq_ghz', 'gpu']

In [387]:
df.drop(columns=drop, inplace = True)

In [388]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 30 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            966 non-null    str    
 1   Price           966 non-null    int64  
 2   Screen Size     966 non-null    float64
 3   Display         966 non-null    int64  
 4   Chipset         966 non-null    int64  
 5   NFC             966 non-null    int64  
 6   ROM             966 non-null    float64
 7   RAM             966 non-null    float64
 8   Battery         966 non-null    float64
 9   Refresh Rate    966 non-null    float64
 10  perf_cores      966 non-null    float64
 11  eff_cores       966 non-null    float64
 12  perf_freq_ghz   966 non-null    float64
 13  eff_freq_ghz    966 non-null    float64
 14  OS_Name         966 non-null    int64  
 15  OS_Version      966 non-null    float64
 16  antutu_11       747 non-null    float64
 17  rear_count      966 non-null    float64
 18  r

In [389]:
# import pandas as pd
# import numpy as np
# from xgboost import XGBRegressor
# from sklearn.impute import SimpleImputer
# from sklearn.model_selection import cross_val_score, KFold

# df = pd.read_csv('full_data_1.csv')

# # ── 1. Định nghĩa features ──────────────────────────────────────────
# drop_cols = ['Unnamed: 0', 'Name', 'Camera_score']
# features = [c for c in df.columns if c not in drop_cols]

# # ── 2. Tách tập train (các dòng đã có Camera_score) ─────────────────
# train = df[df['Camera_score'].notna()].copy()
# X_train = train[features]
# y_train = train['Camera_score']

# # ── 3. Impute cột antutu_11 bị thiếu 22.6% ──────────────────────────
# imputer = SimpleImputer(strategy='median')
# X_train_imp = imputer.fit_transform(X_train)

# # ── 4. Khởi tạo model với best params từ GridSearch ──────────────────
# xgb = XGBRegressor(
#     n_estimators=200,       # số cây
#     learning_rate=0.1,      # tốc độ học
#     max_depth=3,            # độ sâu mỗi cây
#     subsample=0.7,          # % dòng dùng mỗi cây
#     colsample_bytree=0.8,   # % feature dùng mỗi cây
#     random_state=42,
#     verbosity=0
# )

# # ── 5. Train ─────────────────────────────────────────────────────────
# xgb.fit(X_train_imp, y_train)

# # ── 6. Predict toàn bộ dataset ───────────────────────────────────────
# X_all = imputer.transform(df[features])
# df['Camera_score_predicted'] = xgb.predict(X_all)

# # ── 7. Kết hợp: ưu tiên score thực, dùng predicted nếu thiếu ────────
# df['Camera_score_final'] = df['Camera_score'].combine_first(df['Camera_score_predicted'])

In [390]:
# df['Camera_score_final']

In [391]:
# df.to_csv('full_data_3.csv')